# ECG Export for Multimodal Fusion (window-level, EEG-aligned)

Standalone notebook. Reads XDF files (RW, VR, BASELINE) for each participant, runs the original ECG preprocessing unchanged (filter, R-peak detection, RR features, baseline normalisation), and writes window-level CSVs that align with the EEG pipeline so EEG/ECG can be fused at the window granularity.

## What stays the same as the original ECG notebook

- Bandpass filter 0.5–40 Hz, Butterworth order 4, `filtfilt`
- R-peak detection via `nk.ecg_process`
- All 21 HRV features in `compute_rr_features` (incl. `ant.sample_entropy`, `ant.app_entropy`, ratios)
- Baseline normalisation: 30 s windows / 10 s step over BASELINE.xdf → `(b_mean, b_std)` → `(x − b_mean) / b_std`
- Two output folders: `unnorm_*sec_windows/` and `norm_*sec_windows/`
- Marker parsing for both RW (JSON) and VR (`Trial_Start_*`)

## What is changed for EEG alignment

Six targeted changes; preprocessing is untouched:

1. **Per-trial RNG seeding** with `stable_hash(pid, trial_id, world, WINDOW_SEC, MASTER_SEED)` — matches EEG cell 12 byte-for-byte. The `world` argument inside the hash uses EEG's lowercase strings (`'vr'` / `'real'`) — see cell with `extract_windowed_rr`. Output CSVs still use `'VR'` / `'RW'` (uppercase). (Original used one shared RNG advanced across trials.)
2. **`SKIP_SEC = 0`** (was 30). EEG starts windows from sample 0 of every trial; a 30s skip would offset every ECG window by 30s.
3. **Global N** computed across all participants × all worlds × all trials in a first pass, then reused in a second pass. Matches EEG's "shortest valid trial across the dataset" logic.
4. **`WINDOW_SEC` is `5.0` (float)** — `stable_hash` is sensitive to type because it hashes `str(arg)` and `str(5) ≠ str(5.0)`. EEG passes float.
5. **One row per window** with a `window_idx` column (0..N-1, sorted by ascending start). Original was one row per trial.
6. **Batch over participants** — `PARTICIPANTS` list of dicts with full paths to RW/VR/BASELINE XDFs.

## Output layout

```
unnorm_5sec_windows/{PID}_{WORLD}_unnorm_5sec_windows.csv     ← original output
norm_5sec_windows/{PID}_{WORLD}_norm_5sec_windows.csv         ← original output
fusion_inputs/ecg/{PID}_windowed_Normalised_NF_{WORLD}.csv    ← alias for fusion notebook
```

In [1]:
pip install antropy


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Imports

In [2]:
import os
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import pyxdf
import neurokit2 as nk
import antropy as ant
from scipy.signal import butter, filtfilt

## 2. Configuration

`WINDOW_SEC` must equal the `ws` you used for EEG export (`2.0`, `5.0`, or `10.0`). It must be a **float** — `stable_hash` is type-sensitive.

`MASTER_SEED` must equal the EEG notebook's `MASTER_SEED` (= 42).

`SKIP_SEC` is 0 to align with EEG's window start (the original ECG notebook used 30; that's removed here).

In [3]:
# ═══════════════════════════════════════════════════════════
# CONFIG — edit these
# ═══════════════════════════════════════════════════════════
WINDOW_SEC  = 5.0          # MUST be float and equal EEG's ws (2.0 / 5.0 / 10.0)
STEP_SEC    = 2.5          # 50% overlap (= WINDOW_SEC / 2)
SKIP_SEC    = 0            # 0 to align with EEG. EEG starts windows at sample 0.
MASTER_SEED = 42           # MUST equal MASTER_SEED in EEG notebook
ECG_CHANNEL_INDEX = 1      # ECGBIT0 column in OpenSignals stream

OUT_ROOT     = Path('.')
_WS_TAG      = f'{int(WINDOW_SEC)}' if WINDOW_SEC.is_integer() else f'{WINDOW_SEC}'
UNNORM_DIR   = OUT_ROOT / f'unnorm_{_WS_TAG}sec_windows'
NORM_DIR     = OUT_ROOT / f'norm_{_WS_TAG}sec_windows'
FUSION_DIR   = OUT_ROOT / 'fusion_inputs' / 'ecg'
for d in (UNNORM_DIR, NORM_DIR, FUSION_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f'WINDOW_SEC  = {WINDOW_SEC}s   (float — matches EEG type)')
print(f'STEP_SEC    = {STEP_SEC}s     (50% overlap)')
print(f'SKIP_SEC    = {SKIP_SEC}s     (EEG-aligned, was 30 in original)')
print(f'MASTER_SEED = {MASTER_SEED}')
print(f'Output dirs:')
print(f'  unnorm  → {UNNORM_DIR}')
print(f'  norm    → {NORM_DIR}')
print(f'  fusion  → {FUSION_DIR}')

WINDOW_SEC  = 5.0s   (float — matches EEG type)
STEP_SEC    = 2.5s     (50% overlap)
SKIP_SEC    = 0s     (EEG-aligned, was 30 in original)
MASTER_SEED = 42
Output dirs:
  unnorm  → unnorm_5sec_windows
  norm    → norm_5sec_windows
  fusion  → fusion_inputs/ecg


## 3. Participant list

Edit this with full paths to each participant's RW, VR, and BASELINE XDF files. Any missing/None entry is skipped gracefully — the script will still process whichever files are present.

In [4]:
PARTICIPANTS = [
    {
        'pid':      'JAGO02',
        'RW':       '/home/prakunjpratapsingh/deeplearning/converge/realworlddata/sub-JAGO02_ses-RW_task-Default_run-001_eeg.xdf',
        'VR':       '/home/prakunjpratapsingh/deeplearning/converge/vrworlddata/sub-JAGO02_ses-VR_task-Default_run-001_eeg.xdf',
        'BASELINE': '/home/prakunjpratapsingh/deeplearning/converge/Baselinedatarw/sub-JAGO02_ses-Basline_task-Default_run-001_eeg.xdf',
    },
    {
        'pid':      'RUSA03',
        'RW':       '/home/prakunjpratapsingh/deeplearning/converge/realworlddata/sub-RUSA03_ses-RW_task-Default_run-001_eeg.xdf',
        'VR':       '/home/prakunjpratapsingh/deeplearning/converge/vrworlddata/sub-RUSA03_ses-VR_task-Default_run-001_eeg.xdf',
        'BASELINE': '/home/prakunjpratapsingh/deeplearning/converge/Baselinedatarw/sub-RUSA03_ses-BASELINE_RW_task-Default_run-001_eeg.xdf',
    },
    {
        'pid':      'UPDE00',
        'RW':       '/home/prakunjpratapsingh/deeplearning/converge/realworlddata/sub-UPDE00_ses-RW_task-Default_run-001_eeg.xdf',
        'VR':       '/home/prakunjpratapsingh/deeplearning/converge/vrworlddata/sub-UPDE00_ses-VR_task-Default_run-001_eeg.xdf',
        'BASELINE': '/home/prakunjpratapsingh/deeplearning/converge/Baselinedatarw/sub-UPDE00_ses-Baseline_RW_task-Default_run-001_eeg.xdf',
    },
    {
        'pid':      'VIAN01',
        'RW':       '/home/prakunjpratapsingh/deeplearning/converge/realworlddata/sub-VIAN01_ses-RW_task-Default_run-001_eeg.xdf',
        'VR':       '/home/prakunjpratapsingh/deeplearning/converge/vrworlddata/sub-VIAN01_ses-VR_task-Default_run-001_eeg.xdf',
        'BASELINE': '/home/prakunjpratapsingh/deeplearning/converge/Baselinedatarw/sub-VIAN01_ses-Baseline_RW_task-Default_run-001_eeg.xdf',
    },
    {
        'pid':      'WAAR02',
        'RW':       '/home/prakunjpratapsingh/deeplearning/converge/realworlddata/sub-WAA02_ses-RW_task-Default_run-001_eeg.xdf',
        'VR':       '/home/prakunjpratapsingh/deeplearning/converge/vrworlddata/sub-WAAR02_ses-VR_task-Default_run-001_eeg.xdf',
        'BASELINE': '/home/prakunjpratapsingh/deeplearning/converge/Baselinedatarw/sub-WAAR02_ses-Baseline-S001_task-Default_run-001_eeg.xdf',
    },
    
    # add more participants here, e.g.:
    # {
    #     'pid':      'WAAR02',
    #     'RW':       '/path/to/sub-WAAR02_ses-RW_task-Default_run-001_eeg.xdf',
    #     'VR':       '/path/to/sub-WAAR02_ses-VR_task-Default_run-001_eeg.xdf',
    #     'BASELINE': '/path/to/sub-WAAR02_ses-BASELINE_task-Default_run-001_eeg.xdf',
    # },
]
print(f'{len(PARTICIPANTS)} participant(s) configured')

5 participant(s) configured


## 4. `stable_hash` — IDENTICAL to EEG cell 12

This is the function that locks ECG window selection to EEG window selection. Both pipelines must call it with the **same argument types** in the **same order**:

`(pid, trial_id, world, WINDOW_SEC, MASTER_SEED)`

**Important — `world` argument convention:** EEG passes lowercase `'vr'` / `'real'` (set internally in EEG's `trial_df`). For the seed to match, ECG must hash with the same lowercase strings. The translation happens inside `extract_windowed_rr` further down — the CSV output still uses `'VR'` / `'RW'` (uppercase). Only the hash input is mapped.

`stable_hash` is also type-sensitive: `str(5)` ≠ `str(5.0)`. EEG passes `ws` as a float (from `WINDOW_LENGTHS = [2.0, 5.0, 10.0]`), so this notebook keeps `WINDOW_SEC = 5.0` (float).

In [5]:
def stable_hash(*args):
    """Deterministic across machines/OS — uses hashlib MD5, not Python's hash()."""
    s = '|'.join(str(a) for a in args)
    return int(hashlib.md5(s.encode()).hexdigest(), 16) % (2**32)

# Sanity check: confirm the seed for a known input is what EEG produces
# Sanity check: ECG must produce the SAME seed as EEG for the same trial.
# EEG passes lowercase world strings ('vr' / 'real') from its trial_df.
# So the canonical alignment hash uses LOWERCASE world.
demo_seed_real = stable_hash('WAAR02', 'L03T02', 'real', 5.0, 42)
demo_seed_vr   = stable_hash('WAAR02', 'L03T02', 'vr',   5.0, 42)
print(f'stable_hash("WAAR02","L03T02","real",5.0,42) = {demo_seed_real}  (RW trial)')
print(f'stable_hash("WAAR02","L03T02","vr",  5.0,42) = {demo_seed_vr}  (VR trial)')
print('(EEG must produce these exact values for the same trial.)')

stable_hash("WAAR02","L03T02","real",5.0,42) = 316957433  (RW trial)
stable_hash("WAAR02","L03T02","vr",  5.0,42) = 405324243  (VR trial)
(EEG must produce these exact values for the same trial.)


## 5. Bandpass filter — UNCHANGED

In [6]:
def bandpass_filter(signal, fs, lowcut=0.5, highcut=40, order=4):
    nyquist = 0.5 * fs
    b, a = butter(order, [lowcut/nyquist, highcut/nyquist], btype='bandpass')
    return filtfilt(b, a, signal)

## 6. RR / HRV feature extraction — UNCHANGED

All 21 HRV features plus `artifact_ratio`. RR cleanup: clip implausible intervals (<0.3 s or >2.0 s) → NaN → linear interpolation. Same as the original notebook.

In [7]:
def compute_rr_features(rr):
    """Compute HRV features from a raw RR array with internal artifact removal."""
    if len(rr) < 4:
        return None
    mask = (rr < 0.3) | (rr > 2.0)
    artifact_ratio = float(np.mean(mask))
    rr = rr.copy()
    rr[mask] = np.nan
    valid = ~np.isnan(rr)
    if np.sum(valid) < 3:
        return None
    rr = np.interp(np.arange(len(rr)), np.where(valid)[0], rr[valid])

    hr      = 60 / rr
    diff_rr = np.diff(rr)
    rr_mean = np.mean(rr)
    rr_std  = np.std(rr)
    rmssd   = np.sqrt(np.mean(diff_rr**2))
    sd1     = np.std(diff_rr) / np.sqrt(2)
    sd2     = np.sqrt(max(2 * rr_std**2 - sd1**2, 1e-12))

    return {
        'ECG_hr_mean':         np.mean(hr),
        'ECG_hr_std':          np.std(hr),
        'ECG_hr_min':          np.min(hr),
        'ECG_hr_max':          np.max(hr),
        'ECG_rr_mean':         rr_mean,
        'ECG_rr_std':          rr_std,
        'ECG_rmssd':           rmssd,
        'ECG_sdnn':            rr_std,
        'ECG_pnn50':           np.mean(np.abs(diff_rr) > 0.05),
        'ECG_n_beats':         len(rr),
        'ECG_cv_rr':           rr_std / (rr_mean + 1e-6),
        'ECG_sd1':             sd1,
        'ECG_sd2':             sd2,
        'ECG_sd1_sd2':         sd1 / (sd2 + 1e-6),
        'ECG_sampen':          ant.sample_entropy(rr),
        'ECG_apen':            ant.app_entropy(rr),
        'sd1_over_sd2':        sd1 / (sd2 + 1e-6),
        'hr_max_over_hr_min':  np.max(hr) / (np.min(hr) + 1e-6),
        'hr_std_over_hr_mean': np.std(hr) / (np.mean(hr) + 1e-6),
        'hr_range':            np.max(hr) - np.min(hr),
        'rrstd_minus_rmssd':   rr_std - rmssd,
        'artifact_ratio':      artifact_ratio,
    }


FEATURES = [
    'ECG_hr_mean', 'ECG_hr_std', 'ECG_hr_min', 'ECG_hr_max',
    'ECG_rr_mean', 'ECG_rr_std', 'ECG_rmssd', 'ECG_sdnn',
    'ECG_pnn50', 'ECG_n_beats', 'ECG_cv_rr',
    'ECG_sd1', 'ECG_sd2', 'ECG_sd1_sd2',
    'ECG_sampen', 'ECG_apen',
    'sd1_over_sd2', 'hr_max_over_hr_min', 'hr_std_over_hr_mean',
    'hr_range', 'rrstd_minus_rmssd',
]
print(f'{len(FEATURES)} HRV features defined')

21 HRV features defined


## 7. Baseline normalisation — UNCHANGED

Same logic as the original notebook: 30 s windows with 10 s step over the BASELINE XDF, compute features per window, then `mean()` and `std()` across those windows. Trial-window features are then `(x − b_mean) / b_std` per feature.

In [8]:
def compute_baseline_stats(baseline_xdf_path):
    streams, _ = pyxdf.load_xdf(baseline_xdf_path)
    b_stream = None
    for s in streams:
        if 'opensignals' in s['info']['name'][0].lower():
            b_stream = s
            break
    if b_stream is None:
        raise RuntimeError(f'Baseline ECG stream not found in {baseline_xdf_path}')

    b_sig = np.array(b_stream['time_series'])[:, ECG_CHANNEL_INDEX]
    b_fs  = float(b_stream['info']['nominal_srate'][0])

    b_clean   = nk.ecg_clean(b_sig, sampling_rate=b_fs)
    _, b_info = nk.ecg_peaks(b_clean, sampling_rate=b_fs)
    b_rpeaks  = b_info['ECG_R_Peaks']

    B_WIN  = int(30 * b_fs)
    B_STEP = int(10 * b_fs)
    rows = []
    start = 0
    while start + B_WIN <= len(b_clean):
        end = start + B_WIN
        rpk = b_rpeaks[(b_rpeaks >= start) & (b_rpeaks < end)] - start
        if len(rpk) > 5:
            row = compute_rr_features(np.diff(rpk) / b_fs)
            if row:
                rows.append(row)
        start += B_STEP

    bdf = pd.DataFrame(rows)
    return bdf.mean(), bdf.std()


def zscore_normalize(df, b_mean, b_std):
    df_n = df.copy()
    for f in FEATURES:
        if f not in df.columns or f not in b_std.index:
            continue
        std = b_std[f]
        if np.isnan(std) or std == 0:
            df_n[f] = np.nan
        else:
            df_n[f] = (df[f] - b_mean[f]) / std
    return df_n

## 8. Marker parsing & stream finding — UNCHANGED

Both branches handled exactly as the original notebook: RW uses JSON `trial_start`/`trial_end`, VR uses `Trial_Start_*`/`Trial_Stop_*`.

In [9]:
def parse_trials(marker_stream, world):
    marker_times = np.array(marker_stream['time_stamps'])
    trials = []
    active = {}

    if world == 'RW':
        for ts, m in zip(marker_times, marker_stream['time_series']):
            try:
                ev = json.loads(m[0])
            except (json.JSONDecodeError, IndexError):
                continue
            event = ev.get('event', '')
            if event == 'trial_start':
                tid = ev['trial']
                active[tid] = {'trial_id': tid, 'start_time': ts}
            elif event == 'trial_end':
                tid = ev['trial']
                if tid in active:
                    active[tid]['end_time'] = ts
                    trials.append(active.pop(tid))
    elif world == 'VR':
        for ts, m in zip(marker_times, marker_stream['time_series']):
            label = m[0] if isinstance(m[0], str) else str(m[0])
            if label.startswith('Trial_Start_'):
                tid = label.replace('Trial_Start_', '')
                active[tid] = {'trial_id': tid, 'start_time': ts}
            elif label.startswith('Trial_Stop_'):
                tid = label.replace('Trial_Stop_', '')
                if tid in active:
                    active[tid]['end_time'] = ts
                    trials.append(active.pop(tid))
    else:
        raise ValueError(f'Unknown world: {world!r}')
    return trials


def find_ecg_stream(streams):
    for s in streams:
        name  = s['info']['name'][0].lower()
        stype = s['info']['type'][0].lower()
        if 'ecg' in name or 'ecg' in stype or 'opensignals' in name:
            return s
    raise RuntimeError('ECG stream not found')


def find_marker_stream(streams):
    for s in streams:
        name  = s['info']['name'][0].lower()
        stype = s['info']['type'][0].lower()
        if 'button_markers_tablet' in name or 'marker' in stype:
            return s
    raise RuntimeError('Marker stream not found')

## 9. Pass 1 — load every (participant, world), record all trial durations

This pass loads every XDF file, runs the filter and R-peak detection, parses trials, and records every valid trial's duration. **It does NOT pick windows yet** — that has to wait until we know the global minimum trial duration across the whole dataset (so N matches EEG).

In [10]:
def first_pass_load(participants):
    """Returns dict keyed by (pid, world) → dict with ecg_filtered, rpeaks, fs, trials."""
    cache = {}
    all_durations = []

    for p in participants:
        pid = p['pid']
        for world in ('RW', 'VR'):
            xdf_path = p.get(world)
            if xdf_path is None or not os.path.exists(xdf_path):
                print(f'  [{pid} | {world}] file not found, skipped')
                continue

            print(f'  [{pid} | {world}] loading XDF, filtering, detecting R-peaks...')
            streams, _ = pyxdf.load_xdf(xdf_path)
            ecg_stream    = find_ecg_stream(streams)
            marker_stream = find_marker_stream(streams)

            ecg_signal = np.array(ecg_stream['time_series'])[:, ECG_CHANNEL_INDEX]
            fs         = float(ecg_stream['info']['nominal_srate'][0])
            ecg_times  = np.array(ecg_stream['time_stamps'])

            ecg_filtered = bandpass_filter(ecg_signal, fs)
            _, info = nk.ecg_process(ecg_filtered, sampling_rate=fs)
            rpeaks = info['ECG_R_Peaks']

            trials = parse_trials(marker_stream, world)

            # Marker times → ECG sample indices
            for tr in trials:
                tr['start_idx']        = int(np.searchsorted(ecg_times, tr['start_time']))
                tr['end_idx']          = int(np.searchsorted(ecg_times, tr['end_time']))
                tr['eff_start_idx']    = tr['start_idx'] + int(SKIP_SEC * fs)
                tr['eff_duration_sec'] = (tr['end_idx'] - tr['eff_start_idx']) / fs

            valid_trials = [tr for tr in trials if tr['eff_duration_sec'] >= WINDOW_SEC]

            cache[(pid, world)] = {
                'ecg_filtered': ecg_filtered,
                'rpeaks':       rpeaks,
                'fs':           fs,
                'trials':       valid_trials,
            }
            all_durations.extend(tr['eff_duration_sec'] for tr in valid_trials)
            min_here = min((tr['eff_duration_sec'] for tr in valid_trials), default=float('nan'))
            print(f'  [{pid} | {world}] {len(valid_trials)} valid trials, min eff duration {min_here:.1f}s')

    if not all_durations:
        raise RuntimeError('No valid trials loaded across any participant.')

    global_min_duration = min(all_durations)
    N_global = int((global_min_duration - WINDOW_SEC) / STEP_SEC) + 1
    print(f'\n  Global min effective trial duration: {global_min_duration:.1f}s')
    print(f'  → N = {N_global} windows per trial  '
          f'(WINDOW_SEC={WINDOW_SEC}s, STEP_SEC={STEP_SEC}s, 50% overlap)')

    return cache, N_global


print('[Pass 1/2] Loading & detecting R-peaks for every (participant, world)...')
cache, N_GLOBAL = first_pass_load(PARTICIPANTS)
print(f'\nLoaded {len(cache)} (participant, world) entries.')
print(f'Global N = {N_GLOBAL} windows per trial.')

[Pass 1/2] Loading & detecting R-peaks for every (participant, world)...
  [JAGO02 | RW] loading XDF, filtering, detecting R-peaks...
  [JAGO02 | RW] 12 valid trials, min eff duration 60.9s
  [JAGO02 | VR] loading XDF, filtering, detecting R-peaks...


Stream 2: Calculated effective sampling rate 33.5459 Hz is different from specified rate 120.0000 Hz.


  [JAGO02 | VR] 12 valid trials, min eff duration 73.0s
  [RUSA03 | RW] loading XDF, filtering, detecting R-peaks...
  [RUSA03 | RW] 12 valid trials, min eff duration 64.5s
  [RUSA03 | VR] loading XDF, filtering, detecting R-peaks...


Stream 4: Calculated effective sampling rate 33.3890 Hz is different from specified rate 120.0000 Hz.


  [RUSA03 | VR] 12 valid trials, min eff duration 86.3s
  [UPDE00 | RW] loading XDF, filtering, detecting R-peaks...
  [UPDE00 | RW] 12 valid trials, min eff duration 61.8s
  [UPDE00 | VR] loading XDF, filtering, detecting R-peaks...


...

  [UPDE00 | VR] 12 valid trials, min eff duration 92.7s
  [VIAN01 | RW] loading XDF, filtering, detecting R-peaks...
  [VIAN01 | RW] 12 valid trials, min eff duration 69.6s
  [VIAN01 | VR] loading XDF, filtering, detecting R-peaks...


Stream 3: Calculated effective sampling rate 33.4116 Hz is different from specified rate 120.0000 Hz.


  [VIAN01 | VR] 12 valid trials, min eff duration 88.2s
  [WAAR02 | RW] loading XDF, filtering, detecting R-peaks...
  [WAAR02 | RW] 12 valid trials, min eff duration 77.1s
  [WAAR02 | VR] loading XDF, filtering, detecting R-peaks...


Stream 4: Calculated effective sampling rate 33.8676 Hz is different from specified rate 120.0000 Hz.


  [WAAR02 | VR] 12 valid trials, min eff duration 95.8s

  Global min effective trial duration: 60.9s
  → N = 23 windows per trial  (WINDOW_SEC=5.0s, STEP_SEC=2.5s, 50% overlap)

Loaded 10 (participant, world) entries.
Global N = 23 windows per trial.


## 10. Pass 2 — window selection and feature extraction

Now that we know `N_GLOBAL`, walk the cache and:

1. For each trial, build the full pool of overlapping window starts.
2. If the pool is bigger than `N_GLOBAL`, seed `np.random.RandomState(stable_hash(...))` and pick `N_GLOBAL` indices. Sort ascending → `window_idx = 0..N-1`.
3. Extract RR intervals per window, compute the 21 HRV features.
4. Skip windows where features can't be computed (RR < 4 valid beats).

In [11]:
def extract_windowed_rr(tr, rpeaks_all, fs, N_windows, pid, world):
    """Per-trial seeded with stable_hash — matches EEG cell 12."""
    window_samples = int(WINDOW_SEC * fs)
    step_samples   = int(STEP_SEC   * fs)
    eff_start      = tr['eff_start_idx']
    end_idx        = tr['end_idx']

    all_starts = list(range(eff_start, end_idx - window_samples + 1, step_samples))

    if len(all_starts) <= N_windows:
        selected_starts = all_starts                            # use all available
    else:
        # IMPORTANT: hash with EEG's lowercase world strings ('vr'/'real')
        # so the seed matches EEG's stable_hash exactly. The CSV output
        # below still uses 'VR'/'RW' (uppercase) — only the hash input is mapped.
        WORLD_HASH_MAP = {'VR': 'vr', 'RW': 'real'}
        world_for_hash = WORLD_HASH_MAP.get(world, world)
        seed = stable_hash(pid, tr['trial_id'], world_for_hash, WINDOW_SEC, MASTER_SEED)
        rng  = np.random.RandomState(seed)
        chosen = sorted(rng.choice(len(all_starts), size=N_windows, replace=False))
        selected_starts = [all_starts[i] for i in chosen]

    out = []
    for w_start in selected_starts:
        w_end = w_start + window_samples
        w_rpeaks = rpeaks_all[(rpeaks_all >= w_start) & (rpeaks_all < w_end)] - w_start
        rr = np.diff(w_rpeaks) / fs
        out.append((w_start, rr))
    return out


def build_feature_df(pid, world, entry, N_global):
    """One row per window. window_idx = position after sorting by ascending start."""
    fs     = entry['fs']
    rpeaks = entry['rpeaks']
    rows = []
    for tr in entry['trials']:
        windowed = extract_windowed_rr(tr, rpeaks, fs, N_global, pid, world)

        
        for w_idx, (w_start, rr) in enumerate(windowed):
            feats = compute_rr_features(rr)
            if feats is None:
                continue
            rows.append({
                'world':          world,
                'participant_id': pid,
                'trial_id':       tr['trial_id'],
                'level':          int(tr['trial_id'][1:3]),
                'trial':          tr['trial_id'][3:],
                'window_idx':     w_idx,
                **feats,
            })
    return pd.DataFrame(rows)

print('Window extraction + feature computation functions ready.')

Window extraction + feature computation functions ready.


## 11. Save the CSVs (unnorm + norm + fusion alias)

For each `(participant, world)`:

- Compute baseline stats once per participant (cached, reused across RW + VR).
- Build the window-level feature DataFrame.
- Save **unnormalised** copy to `unnorm_*sec_windows/{PID}_{WORLD}_unnorm_*.csv` (original output).
- Save **baseline-normalised** copy to `norm_*sec_windows/{PID}_{WORLD}_norm_*.csv` (original output).
- Save a **fusion alias** copy to `fusion_inputs/ecg/{PID}_windowed_Normalised_NF_{WORLD}.csv` so the multimodal fusion notebook can find it.

In [15]:
# ═══════════════════════════════════════════════════════════
# ECG → TRIAL LEVEL (MEAN AGGREGATION)
# ═══════════════════════════════════════════════════════════

import os

OUT_DIR = "fusion_inputs/ecg_trial_level"
os.makedirs(OUT_DIR, exist_ok=True)

STEP_SEC = STEP_SEC  # already defined
WINDOW = WINDOW_SEC

rows = []

for (pid, world), df in [
    ((pid, world), build_feature_df(pid, world, cache[(pid, world)], N_GLOBAL))
    for (pid, world) in cache
]:

    if df.empty:
        continue

    for trial_id, g in df.groupby("trial_id"):

        try:
            level = int(trial_id[1:3])
            trial = trial_id[3:]
        except:
            continue

        N = len(g)

        if N == 0:
            continue

        used_duration = (N - 1) * STEP_SEC + WINDOW

        row = {
            "world": world,
            "participant_id": pid,
            "trial_id": trial_id,
            "level": level,
            "trial": trial,
            "n_valid_windows": N,
            "used_duration": used_duration,
        }

        #  MEAN aggregation (NaNs automatically ignored)
        for f in FEATURES:
            if f in ['ECG_sampen', 'ECG_apen']:
                continue  # already removed

            if f in g.columns:
                val = g[f].mean()

                if np.isnan(val):
                    print(f" NaN after mean → {pid} | {world} | {trial_id} | {f}")

                row[f] = float(val)

        rows.append(row)

# ───────── SAVE ─────────
out_df = pd.DataFrame(rows)

for (pid, world), g in out_df.groupby(["participant_id", "world"]):
    fname = f"{pid}_windowed_Normalised_NF_{world}.csv"
    g = g.sort_values("trial_id")
    g.to_csv(os.path.join(OUT_DIR, fname), index=False)

print("\n ECG trial-level export complete")
print("Shape:", out_df.shape)
print("Saved in:", OUT_DIR)


 ECG trial-level export complete
Shape: (120, 26)
Saved in: fusion_inputs/ecg_trial_level
